# Phase 0 Sanity Check

Verifies the rebuilt `src/data_prep.py` pipeline output is consistent and that the
existing Ridge baseline still works as expected. If everything here passes, Phase 0
is closed and Phase 1 modeling can begin.

**Sections**
1. Load + NaN sanity
2. Splits coverage and disjointness
3. Feature-group partition + pairwise disjointness
4. Cluster distribution and composition
5. Ridge baseline — official val, with and without `cluster_id`


In [1]:
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from src.config import (
    CLUSTER_NAMES,
    FEATURE_GROUPS,
    K_CLUSTERS,
    PROCESSED_PATH,
    get_feature_group,
    get_splits,
)


## 1. Load + NaN sanity

The pipeline already runs internal assertions (one-hot sum, `played_*` in {0,1},
`cluster_id` complete, lag leakage spot-check). This cell loads the artifact and
adds NaN visibility — anything that slipped past the explicit asserts.


In [2]:
df = pd.read_csv(PROCESSED_PATH)
print(f"Shape: {df.shape}")

nan_counts = df.isna().sum()
nan_counts = nan_counts[nan_counts > 0]
if len(nan_counts):
    print("\nColumns with NaN:")
    print(nan_counts.to_string())
else:
    print("No NaN values anywhere.")


Shape: (75075, 56)
No NaN values anywhere.


In [3]:
# Hard guarantees for downstream code
no_nan = ["played_60min", "played_any", "cluster_id"] + get_feature_group("POSITION")
for c in no_nan:
    assert df[c].isna().sum() == 0, f"{c} has NaN"
print("Critical no-NaN columns: clean.")


Critical no-NaN columns: clean.


## 2. Splits

`get_splits` already asserts disjointness and full coverage. We re-print the sizes
here so they're visible in the notebook output. Expected magnitudes:
- Train (21/22 + 22/23): ~46k
- Val (23/24, GW 2-33):   ~23k
- Test (23/24, GW 34-38): ~5k


In [4]:
train, val, test = get_splits(df)
total = len(df)
print(f"Train: {train.sum():>6,}  ({train.sum()/total:.1%})")
print(f"Val:   {val.sum():>6,}  ({val.sum()/total:.1%})")
print(f"Test:  {test.sum():>6,}  ({test.sum()/total:.1%})")
print(f"Total: {total:>6,}")


Train: 46,991  (62.6%)
Val:   23,825  (31.7%)
Test:   4,259  (5.7%)
Total: 75,075


## 3. Feature-group partition

Two checks:
1. **Coverage** — every column in `df` belongs to some named group. (Same check
   `data_prep.main()` runs; redundant here but cheap and explicit.)
2. **Pairwise disjoint** — no column lives in two groups. The pipeline assertion
   doesn't catch this on its own (set-union swallows duplicates).


In [5]:
# (1) Coverage
all_grouped = set().union(*FEATURE_GROUPS.values())
unaccounted = set(df.columns) - all_grouped
assert not unaccounted, f"unaccounted columns: {unaccounted}"

# (2) Pairwise disjoint
groups = {g: set(get_feature_group(g)) for g in FEATURE_GROUPS}
for a, b in combinations(groups, 2):
    overlap = groups[a] & groups[b]
    assert not overlap, f"{a} and {b} overlap on: {overlap}"

# Summary
for name, cols in FEATURE_GROUPS.items():
    print(f"{name:18} {len(cols):>3} cols")
total_grouped = sum(len(c) for c in FEATURE_GROUPS.values())
print(f"{'TOTAL':18} {total_grouped:>3} cols  (df has {len(df.columns)})")


IDS                  7 cols
TARGET               1 cols
STAGE1_LABELS        2 cols
BASE_PREKICKOFF     10 cols
POSITION             4 cols
XP                   1 cols
ICT_BLOCK           12 cols
OTHER_LAGS          18 cols
CLUSTER              1 cols
TOTAL               56 cols  (df has 56)


## 4. Cluster distribution and composition

Three things to look at:
- **Sizes** — should roughly match the names in `CLUSTER_NAMES`.
- **Mean target** — premium attackers (cluster 2) should produce more points
  per GW; unclassified (cluster 4) should be lowest because it's mostly
  rotation/cup-of-coffee players.
- **Positional composition** — cluster 0 should be GK/DEF dominant;
  attacker-heavy clusters should show MID/FWD concentration. The whole point
  of excluding position from the K-means features is that we expect *partial*
  positional overlap, not 100%.


In [6]:
cluster_summary = (
    df.groupby("cluster_id")
      .agg(n_rows=("total_points", "size"),
           mean_points=("total_points", "mean"),
           mean_value=("value", "mean"),
           pct_played_60=("played_60min", "mean"))
      .round(3)
)
cluster_summary["name"] = cluster_summary.index.map(CLUSTER_NAMES)
print(cluster_summary.to_string())


            n_rows  mean_points  mean_value  pct_played_60                 name
cluster_id                                                                     
0            16467        1.795      46.184          0.534   Defensive starters
1            17107        0.912      48.885          0.179  Rotational / fringe
2             4141        3.689      82.888          0.616    Premium attackers
3            10090        2.428      54.410          0.564    Creative regulars
4            27270        0.257      44.508          0.071         Unclassified


In [7]:
# Positional share within each cluster (rows are clusters, cols are positions)
pos_cols = get_feature_group("POSITION")
pos_pct = df.groupby("cluster_id")[pos_cols].mean().round(3)
pos_pct.index = pos_pct.index.map(CLUSTER_NAMES)
print(pos_pct.to_string())


                     pos_DEF  pos_FWD  pos_GK  pos_MID
cluster_id                                            
Defensive starters     0.687    0.000   0.189    0.123
Rotational / fringe    0.219    0.188   0.000    0.593
Premium attackers      0.025    0.379   0.000    0.596
Creative regulars      0.199    0.101   0.000    0.699
Unclassified           0.292    0.136   0.193    0.378


## 5. Ridge sanity check

Two runs on the **official** val set (23/24 GW 2-33). The test set (GW 34-38) is
not touched.

**Caveats on the comparison to the prior 0.61 baseline:**
- Prior baseline was on the old 51-col CSV with the `pos_GKP` bug, full 23/24 as
  val (no test set carve-out). This run is on the 56-col CSV with that fix and a
  smaller val set. A few hundredths of difference is expected and not
  diagnostic of a pipeline error.
- Both runs use the same alpha grid; we report the full sweep, not just the
  best — useful as the start of the model table the project rubric requires
  (≥76 models across approaches/transformations/regularizations).
- `opponent_team` is fed in as a continuous integer (1-20). That's not great
  modeling but matches the prior baseline. Phase 1 should reconsider — likely
  one-hot or merge in fixture difficulty rating.


In [8]:
def ridge_sweep(X_train, y_train, X_val, y_val,
                alphas=(0.01, 0.1, 1.0, 10.0, 100.0)):
    """Ridge val R² across an alpha grid. Scaler fit on train only."""
    scaler = StandardScaler().fit(X_train)
    Xt, Xv = scaler.transform(X_train), scaler.transform(X_val)
    rows = []
    for a in alphas:
        m = Ridge(alpha=a).fit(Xt, y_train)
        rows.append({
            "alpha": a,
            "train_r2": m.score(Xt, y_train),
            "val_r2": m.score(Xv, y_val),
        })
    return pd.DataFrame(rows).round(4)


features_no_cluster = (
    get_feature_group("BASE_PREKICKOFF")
    + get_feature_group("XP")
    + get_feature_group("ICT_BLOCK")
    + get_feature_group("OTHER_LAGS")
    + get_feature_group("POSITION")
)
print(f"Feature count (no cluster): {len(features_no_cluster)}")


Feature count (no cluster): 45


In [9]:
# Run 1 — no cluster_id
y_tr = df.loc[train, "total_points"]
y_v  = df.loc[val,   "total_points"]

X_tr = df.loc[train, features_no_cluster].astype(float)
X_v  = df.loc[val,   features_no_cluster].astype(float)

print(f"Train: {X_tr.shape}, Val: {X_v.shape}")
res_no_cluster = ridge_sweep(X_tr, y_tr, X_v, y_v)
print(res_no_cluster.to_string(index=False))


Train: (46991, 45), Val: (23825, 45)
 alpha  train_r2  val_r2
  0.01    0.5981  0.5989
  0.10    0.5981  0.5989
  1.00    0.5981  0.5989
 10.00    0.5981  0.5989
100.00    0.5979  0.5985


In [10]:
# Run 2 — + cluster_id one-hot (drop cluster_4 / unclassified as reference)
cluster_dummies = pd.get_dummies(df["cluster_id"], prefix="cluster").astype(int)
cluster_feats = [c for c in cluster_dummies.columns if c != "cluster_4"]
df_aug = pd.concat([df, cluster_dummies], axis=1)

features_with_cluster = features_no_cluster + cluster_feats
X_tr2 = df_aug.loc[train, features_with_cluster].astype(float)
X_v2  = df_aug.loc[val,   features_with_cluster].astype(float)

print(f"Feature count: {len(features_no_cluster)} -> {len(features_with_cluster)}")
res_with_cluster = ridge_sweep(X_tr2, y_tr, X_v2, y_v)
print(res_with_cluster.to_string(index=False))

delta = res_with_cluster['val_r2'].max() - res_no_cluster['val_r2'].max()
print(f"\nBest val R²:  no cluster  = {res_no_cluster['val_r2'].max():.4f}")
print(f"            + cluster   = {res_with_cluster['val_r2'].max():.4f}")
print(f"            delta       = {delta:+.4f}")


Feature count: 45 -> 49
 alpha  train_r2  val_r2
  0.01    0.6006  0.5933
  0.10    0.6006  0.5933
  1.00    0.6006  0.5933
 10.00    0.6006  0.5932
100.00    0.6005  0.5929

Best val R²:  no cluster  = 0.5989
            + cluster   = 0.5933
            delta       = -0.0056


## Closeout

**What this notebook proves**
- Pipeline output shape is exactly what `src/config` expects (74 raw cols
  reduced to 56 after dropping same-GW post-kickoff stats).
- Splits, feature groups, and cluster IDs are all internally consistent.
- A linear baseline still trains cleanly on the new dataset.

**What this notebook does *not* prove (out of scope for Phase 0)**
- That the cluster_id contribution is meaningful. Whatever delta you see above
  is one number on one model with a fixed alpha grid. Phase 1 should re-test
  cluster value with PCA-projected features and with hurdle-stage modeling.
- That `played_60min` is the right hurdle threshold. The label rate (~29%)
  suggests a reasonable positive class size; whether 60min vs `played_any`
  vs 45min gives the cleanest two-stage decomposition is an empirical
  question Phase 1 answers.
- That the new val R² matches 0.61 ± 0.02. As noted above, the comparison
  isn't apples-to-apples. Treat the result as a plausibility check.

**Carry-forward concerns**
- `opponent_team` as continuous int — fix in Phase 1.
- Cluster 4 (unclassified) at 36% of rows is large. The cluster feature
  partly encodes "fringe player" alongside "archetype." Note in writeup.
